<a href="https://colab.research.google.com/github/PhilippStahlbergGit/DAT251_project_library_app/blob/testing/DAT251_recommendations_system_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experimental retrival of some data, to see the output:


Retrival of data
From: https://www.kaggle.com/datasets/bahramjannesarr/goodreads-book-datasets-10m
Docs: https://goodreads.readthedocs.io/en/latest/

In [ ]:
!pip install faiss-cpu
#!pip install # faiss seems to not work?

# Import dependencies

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]

import kagglehub
from kagglehub import KaggleDatasetAdapter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import faiss

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import RobustScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import FeatureHasher
from sklearn.decomposition import TruncatedSVD, PCA


# Load and preprocessing of data

In [ ]:
# Dataset config
DATASET = "bahramjannesarr/goodreads-book-datasets-10m"
FILES = ["book1-100k.csv", "book1000k-1100k.csv"]

# Load datasets
dfs = [
    kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        DATASET,
        f,
        pandas_kwargs={"usecols": ["Id", "Name", "Authors", "pagesNumber", "PublishYear", "Rating", "RatingDistTotal"]}
    )
    for f in FILES
]
df = pd.concat(dfs, ignore_index=True)

missing_count = df['Authors'].isna().sum()
print(f"Number of books with missing authors: {missing_count}")
missing_authors = df[df['Authors'].isna() | (df['Authors'] == "")]
print(missing_authors)
print(df['Authors'].value_counts(dropna=False).head(10))

# Map authors to IDs
df['AuthorId'], unique_authors = pd.factorize(df['Authors'])
author_lookup = pd.DataFrame({
    'AuthorId': df['AuthorId'].unique(),
    'AuthorName': unique_authors
})

# Clean and compute RatingRatio
df['RatingDistTotal'] = df['RatingDistTotal'].str.replace('total:', '', regex=False).astype(int)
df['RatingRatio'] = np.where(df['Rating'] == 0, 0, df['RatingDistTotal'] / df['Rating'])

# Drop unused columns
df.drop(columns=['Authors', 'RatingDistTotal', 'Rating'], inplace=True)

# Lookup dicts
bookid_to_name = dict(zip(df["Id"], df["Name"]))
authorid_to_name = dict(zip(author_lookup["AuthorId"], author_lookup["AuthorName"]))

/tmp/ipython-input-12014/2497659195.py:7: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  kagglehub.load_dataset(


Using Colab cache for faster access to the 'goodreads-book-datasets-10m' dataset.


/tmp/ipython-input-12014/2497659195.py:7: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  kagglehub.load_dataset(


Using Colab cache for faster access to the 'goodreads-book-datasets-10m' dataset.
Number of books with missing authors: 0
Empty DataFrame
Columns: [Id, Name, pagesNumber, RatingDistTotal, PublishYear, Authors, Rating]
Index: []
Authors
Anonymous              181
William Shakespeare    161
Carolyn Keene          145
Stephen King           129
Isaac Asimov           117
Rumiko Takahashi       114
Piers Anthony          108
Harold Bloom           107
J.R.R. Tolkien         103
C.S. Lewis              98
Name: count, dtype: int64


KeyError: 'authors'

In [ ]:
#df.iloc[0]
#df.columns
#df = df.drop(columns=['Name'])
#df = df.drop(columns=['RatingRatio'])

#Vet ikke om vi trenger denne lenger
#book_name_lookup = df[['Id', 'Name']] # NOTE: These Id's are actually the Id's of goodread (for the API) remember to not mix this up!!


# Feature engineering

In [ ]:
# Numeric features
df["ratingRatio_log"] = np.log1p(df["RatingRatio"])
num_cols = ["pagesNumber", "PublishYear", "ratingRatio_log"]
robust_scaler = RobustScaler()
X_num = robust_scaler.fit_transform(df[num_cols].astype(float)).astype(np.float32)
joblib.dump(robust_scaler, "robust_scaler.joblib")

# Title features
MAX_TITLE_FEATURES = 5000
tfidf = TfidfVectorizer(stop_words="english", max_features=MAX_TITLE_FEATURES, ngram_range=(1,2), min_df=2)
X_title_sparse = tfidf.fit_transform(df["Name"].fillna(""))
svd = TruncatedSVD(n_components=256, random_state=42)
X_title = svd.fit_transform(X_title_sparse).astype(np.float32)

# Author features
N_AUTHOR_BUCKETS = 512
AUTHOR_WEIGHT = 0.1
TITLE_WEIGHT = 0.3
hasher = FeatureHasher(n_features=N_AUTHOR_BUCKETS, input_type="string", alternate_sign=False)
author_tokens = df["AuthorId"].astype(str).apply(lambda a: [f"author={a}"]).tolist()
X_author = hasher.transform(author_tokens).astype(np.float32).toarray() * AUTHOR_WEIGHT

# Combine features and normalize
X_title_scaled = X_title * TITLE_WEIGHT
X = np.hstack([X_num, X_author, X_title_scaled]).astype(np.float32)
faiss.normalize_L2(X)

#Build FAISS index

In [ ]:
# --- Define seed query vector first
i = 0
q = X[i:i+1]  # shape (1, d)

# --- Flat index (simple test)
d = X.shape[1]  # vector dimension
index = faiss.IndexFlatIP(d)  # cosine via IP because we normalized
index.add(X)
D_flat, I_flat = index.search(q, 10)  # top-10 similar for flat index
print("Flat Index results:")
print(D_flat[0], I_flat[0])

# --- IVF index for larger datasets
nlist = 1024  # number of clusters
quantizer = faiss.IndexFlatIP(d)
ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

# Train IVF on a sample
train_n = min(50000, X.shape[0])
ivf.train(X[:train_n])
ivf.add(X)
ivf.nprobe = 16

# Query IVF index
D_ivf, I_ivf = ivf.search(q, 10)
print("IVF Index results:")
print(D_ivf[0], I_ivf[0])

# Optional: save IVF index
faiss.write_index(ivf, "books_ivf.index")

#Vectorize and recommend functions

In [ ]:
def vectorize_one_book(book_df: pd.DataFrame, robust_scaler) -> np.ndarray:
    row = book_df.iloc[0]

    # Numeric features
    pages = float(row["pagesNumber"])
    year = float(row["PublishYear"])
    rr = float(row["RatingRatio"])
    rr_log = np.log1p(rr)

    X_num = robust_scaler.transform(
        np.array([[pages, year, rr_log]], dtype=np.float32)
    ).astype(np.float32)

    # Author feature
    author_id = str(int(row["AuthorId"]))
    X_author = (
        hasher.transform([[f"author={author_id}"]])
        .astype(np.float32)
        .toarray()
        * AUTHOR_WEIGHT
    )

    # Title feature
    X_title = (
        svd.transform(tfidf.transform([row["Name"]]))
        .astype(np.float32)
        * TITLE_WEIGHT
    )

    # Combine
    q = np.hstack([X_num, X_author, X_title]).astype(np.float32)
    faiss.normalize_L2(q)

    return q


def recommend_from_payload(book_df, index, robust_scaler, k=10):
    book_df = book_df.reset_index(drop=True)
    q = vectorize_one_book(book_df, robust_scaler)
    D, I = index.search(q, k)

    results = []
    for row_idx, score in zip(I[0], D[0]):
        b_id = int(df.iloc[row_idx]["Id"])
        a_id = int(df.iloc[row_idx]["AuthorId"])

        results.append({
            "title": bookid_to_name.get(b_id),
            "author": authorid_to_name.get(a_id),
            "score": float(score)
        })

    return results

# Using the functions

In [ ]:
# Single book
sample_book = pd.DataFrame({
    "pagesNumber": [300],
    "PublishYear": [2020],
    "AuthorId": [0],
    "RatingRatio": [1000],
    "Name": ["Introduction to Machine Learning"]
})
recommend_from_payload(sample_book, index, robust_scaler, k=10)

# Multiple seed books
seed_indices = [0, 50, 100]
for i in seed_indices:
    print(f"\nSeed book: {bookid_to_name[df.iloc[i]['Id']]}")
    recommend_from_payload(df.iloc[[i]], index, robust_scaler, k=5)

In [ ]:
results = recommend_from_payload(sample_book, index, robust_scaler, k=10)

for r in results:
    print(f"- {r['title']} by {r['author']} (Score: {r['score']:.4f})")

In [ ]:
def print_recommendations(seed_row: int, k: int = 10):
    q = X[seed_row:seed_row+1]
    D, I = index.search(q, k + 1)

    seed_book_id = int(df.iloc[seed_row]["Id"])
    seed_name = bookid_to_name.get(seed_book_id, "Unknown Book")

    print(f"Seed: {seed_name}\n")
    print("Recommended Books:\n")

    for row_idx, sim in zip(I[0][1:], D[0][1:]):  # skip self
        book_id = int(df.iloc[row_idx]["Id"])
        author_id = int(df.iloc[row_idx]["AuthorId"])

        book_name = bookid_to_name.get(book_id, "Unknown Book")
        author_name = authorid_to_name.get(author_id, "Unknown Author")

        print(f"- {book_name} by {author_name} (Score: {sim:.4f})")

In [ ]:
def plot_similarity_hist(D_result):
    """
    Plots a histogram of similarity scores from a FAISS search.

    Args:
        D_result: numpy array of shape (1, k) returned by index.search()
    """
    sims = D_result[0]  # top-k similarities for the query
    plt.figure(figsize=(6,4))
    plt.hist(sims, bins=20)
    plt.title("Similarity Scores")
    plt.xlabel("Cosine similarity")
    plt.ylabel("Count")
    plt.show()

# Example usage with flat index results:
plot_similarity_hist(D_flat)

# Or if you want IVF results:
# plot_similarity_hist(D_ivf)

In [ ]:
# Reduce to 2D for visualization
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)

def visualize_recommendations(seed_row: int, k: int = 10):
    q = X[seed_row:seed_row+1]
    D, I = index.search(q, k + 1)

    rec_rows = I[0][1:]  # remove self
    seed_point = X_2d[seed_row]
    rec_points = X_2d[rec_rows]

    plt.figure(figsize=(8, 8))

    # Plot all books
    plt.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.2)

    # Plot recommendations
    plt.scatter(rec_points[:, 0], rec_points[:, 1], s=100)

    # Plot seed
    plt.scatter(seed_point[0], seed_point[1], s=300)

    plt.title("Book Recommendation Visualization (PCA Projection)")
    plt.show()

In [ ]:
visualize_recommendations(seed_row=0, k=10)

In [ ]:
def recommend_by_row(seed_row: int, k: int = 10):
    q = X[seed_row:seed_row+1]
    D, I = index.search(q, k + 1)

    seed_id = int(df.iloc[seed_row]["Id"])
    seed_title = bookid_to_name.get(seed_id)

    print(f"\nInput Book: {seed_title}\n")
    print("Recommended Books:\n")

    for row_idx, score in zip(I[0][1:], D[0][1:]):  # skip self
        book_id = int(df.iloc[row_idx]["Id"])
        author_id = int(df.iloc[row_idx]["AuthorId"])

        print(
            f"- {bookid_to_name.get(book_id)} "
            f"by {authorid_to_name.get(author_id)} "
            f"(Score: {score:.4f})"
        )

In [ ]:
def recommend_by_title(title: str, k: int = 10):
    matches = df[df["Name"].str.lower() == title.lower()]

    if len(matches) == 0:
        print("Book not found in dataset.")
        return

    seed_row = matches.index[0]
    recommend_by_row(seed_row, k)

In [ ]:
recommend_by_title("The Hobbit", k=10)

In [ ]:
def recommend_from_book_list(book_list: pd.DataFrame, k: int = 10):
    """
    book_list: DataFrame with multiple books (same format as sample_book)
    Returns recommendations based on averaged user profile
    """

    # Vectorize each owned book
    vectors = []
    for i in range(len(book_list)):
        vec = vectorize_one_book(book_list.iloc[[i]])
        vectors.append(vec)

    # Create user profile vector (mean)
    user_vector = np.mean(np.vstack(vectors), axis=0, keepdims=True)

    # Normalize for cosine similarity
    faiss.normalize_L2(user_vector)

    # Search FAISS
    D, I = index.search(user_vector, k + len(book_list))

    print("\nUser owns:\n")
    for name in book_list["Name"]:
        print(f"- {name}")

    print("\nRecommended Books:\n")

    recommended = 0
    owned_titles = set(book_list["Name"].str.lower())

    for row_idx, score in zip(I[0], D[0]):
        book_id = int(df.iloc[row_idx]["Id"])
        author_id = int(df.iloc[row_idx]["AuthorId"])
        title = bookid_to_name.get(book_id)

        # Skip already owned books
        if title.lower() in owned_titles:
            continue

        print(
            f"- {title} "
            f"by {authorid_to_name.get(author_id)} "
            f"(Score: {score:.4f})"
        )

        recommended += 1
        if recommended >= k:
            break

In [ ]:
my_books = pd.DataFrame({
    "Name": [
        "The Hobbit",
        "The Lord of the Rings",
        "Silmarillion"
    ],
    "pagesNumber": [310, 1178, 365],
    "PublishYear": [1937, 1954, 1977],
    "AuthorId": [0, 0, 0],
    "RatingRatio": [5000, 8000, 3000]
})

recommend_from_book_list(my_books, k=10)

In [ ]:
import numpy as np
import pandas as pd
import faiss

# Vectorize a single book into the feature space
def vectorize_one_book(book_df: pd.DataFrame) -> np.ndarray:
    row = book_df.iloc[0]

    # Numeric features
    pages = float(row["pagesNumber"])
    year = float(row["PublishYear"])
    rr = float(row["RatingRatio"])
    rr_log = np.log1p(rr)
    X_num = robust_scaler.transform(np.array([[pages, year, rr_log]], dtype=np.float32))

    # Author feature
    author_id = str(int(row["AuthorId"]))
    X_author = hasher.transform([[f"author={author_id}"]]).astype(np.float32).toarray() * AUTHOR_WEIGHT

    # Title feature
    X_title = svd.transform(tfidf.transform([row["Name"]])).astype(np.float32) * TITLE_WEIGHT

    # Combine
    q = np.hstack([X_num, X_author, X_title]).astype(np.float32)
    faiss.normalize_L2(q)
    return q

# Recommend books based on a DataFrame of owned books
def recommend_from_book_list(book_list: pd.DataFrame, k: int = 10):
    """
    book_list: DataFrame with columns ['Name', 'pagesNumber', 'PublishYear', 'AuthorId', 'RatingRatio']
    Returns top-k recommended books not in the input list
    """
    vectors = [vectorize_one_book(book_list.iloc[[i]]) for i in range(len(book_list))]

    # Average vector → user profile
    user_vector = np.mean(np.vstack(vectors), axis=0, keepdims=True)
    faiss.normalize_L2(user_vector)

    # Search FAISS
    D, I = index.search(user_vector, k + len(book_list))  # extra to skip owned books

    owned_titles = set(book_list["Name"].str.lower())
    recommended = 0

    print("\nUser owns:\n")
    for name in book_list["Name"]:
        print(f"- {name}")

    print("\nRecommended Books:\n")
    for row_idx, score in zip(I[0], D[0]):
        book_id = int(df.iloc[row_idx]["Id"])
        author_id = int(df.iloc[row_idx]["AuthorId"])
        title = bookid_to_name.get(book_id)

        if title.lower() in owned_titles:
            continue

        print(f"- {title} by {authorid_to_name.get(author_id)} (Score: {score:.4f})")
        recommended += 1
        if recommended >= k:
            break

TESTING BELOW.


#All necessary code in one block

In [ ]:
# ==============================
# INSTALL & IMPORTS
# ==============================

!pip install faiss-cpu

import kagglehub
from kagglehub import KaggleDatasetAdapter

import numpy as np
import pandas as pd
import faiss
from sklearn.preprocessing import RobustScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import FeatureHasher
from sklearn.decomposition import TruncatedSVD


# ==============================
# LOAD DATA
# ==============================

DATASET = "bahramjannesarr/goodreads-book-datasets-10m"
FILES = ["book1-100k.csv", "book1000k-1100k.csv"]

dfs = [
    kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        DATASET,
        f,
        pandas_kwargs={
            "usecols": [
                "Id", "Name", "Authors",
                "pagesNumber", "PublishYear",
                "Rating", "RatingDistTotal"
            ]
        }
    )
    for f in FILES
]

df = pd.concat(dfs, ignore_index=True)

# Author IDs
df["AuthorId"], unique_authors = pd.factorize(df["Authors"])

# Clean rating
df["RatingDistTotal"] = df["RatingDistTotal"].str.replace(
    "total:", "", regex=False
).astype(int)

df["RatingRatio"] = np.where(
    df["Rating"] == 0,
    0,
    df["RatingDistTotal"] / df["Rating"]
)

df.drop(columns=["Authors", "RatingDistTotal", "Rating"], inplace=True)

# Lookup dictionaries
bookid_to_name = dict(zip(df["Id"], df["Name"]))
authorid_to_name = dict(zip(df["AuthorId"], unique_authors))


# ==============================
# FEATURE ENGINEERING
# ==============================

# ----- Numeric features -----
df["ratingRatio_log"] = np.log1p(df["RatingRatio"])

num_cols = ["pagesNumber", "PublishYear", "ratingRatio_log"]
scaler = RobustScaler()
X_num = scaler.fit_transform(df[num_cols]).astype(np.float32)

# ----- Title features -----
TITLE_WEIGHT = 0.3

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

X_title_sparse = tfidf.fit_transform(df["Name"].fillna(""))

svd = TruncatedSVD(n_components=256, random_state=42)
X_title = svd.fit_transform(X_title_sparse).astype(np.float32)
X_title *= TITLE_WEIGHT

# ----- Author features -----
AUTHOR_WEIGHT = 0.1

hasher = FeatureHasher(
    n_features=512,
    input_type="string",
    alternate_sign=False
)

author_tokens = df["AuthorId"].astype(str).apply(
    lambda a: [f"author={a}"]
).tolist()

X_author = hasher.transform(author_tokens).toarray().astype(np.float32)
X_author *= AUTHOR_WEIGHT

# ----- Combine features -----
X = np.hstack([X_num, X_author, X_title]).astype(np.float32)

# Normalize for cosine similarity
faiss.normalize_L2(X)


# ==============================
# BUILD FAISS INDEX
# ==============================

d = X.shape[1]
index = faiss.IndexFlatIP(d)
index.add(X)


# ==============================
# RECOMMENDATION FUNCTION
# ==============================

def recommend_by_title(title: str, k: int = 10):
    matches = df[df["Name"].str.lower() == title.lower()]

    if len(matches) == 0:
        print("Book not found in dataset.")
        return

    seed_row = matches.index[0]
    q = X[seed_row:seed_row+1]

    D, I = index.search(q, k + 1)

    seed_id = int(df.iloc[seed_row]["Id"])
    print(f"\nInput Book: {bookid_to_name[seed_id]}\n")
    print("Recommended Books:\n")

    for row_idx, score in zip(I[0][1:], D[0][1:]):  # skip itself
        book_id = int(df.iloc[row_idx]["Id"])
        author_id = int(df.iloc[row_idx]["AuthorId"])

        print(
            f"- {bookid_to_name.get(book_id)} "
            f"by {authorid_to_name.get(author_id)} "
            f"(Score: {score:.4f})"
        )


# ==============================
# EXAMPLE
# ==============================

recommend_by_title("The Hobbit", k=10)

# Fiks det slik at input blir bøkene du har hjemme i biblioteket ditt. Inputet skal være pandas dataframe.

In [ ]:
# ==============================
# INSTALL & IMPORTS
# ==============================
!pip install faiss-cpu

import kagglehub
from kagglehub import KaggleDatasetAdapter

import numpy as np
import pandas as pd
import faiss
from sklearn.preprocessing import RobustScaler
from sklearn.feature_extraction.text import TfidfVectorizer, FeatureHasher
from sklearn.decomposition import TruncatedSVD

# ==============================
# LOAD DATA
# ==============================
DATASET = "bahramjannesarr/goodreads-book-datasets-10m"
FILES = ["book1-100k.csv", "book1000k-1100k.csv"]

dfs = [
    kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        DATASET,
        f,
        pandas_kwargs={"usecols": ["Id", "Name", "Authors", "pagesNumber", "PublishYear", "Rating", "RatingDistTotal"]}
    )
    for f in FILES
]

df = pd.concat(dfs, ignore_index=True)

# Handle missing authors
df["Authors"] = df["Authors"].fillna("Unknown Author")

# Author IDs
df["AuthorId"], unique_authors = pd.factorize(df["Authors"])

# Clean rating
df["RatingDistTotal"] = df["RatingDistTotal"].str.replace("total:", "", regex=False).astype(int)
df["RatingRatio"] = np.where(df["Rating"] == 0, 0, df["RatingDistTotal"] / df["Rating"])
df.drop(columns=["Authors", "RatingDistTotal", "Rating"], inplace=True)

# Lookup dictionaries
bookid_to_name = dict(zip(df["Id"], df["Name"]))
authorid_to_name = dict(zip(df["AuthorId"], unique_authors))

# ==============================
# FEATURE ENGINEERING
# ==============================
TITLE_WEIGHT = 0.3
AUTHOR_WEIGHT = 0.1

# Numeric features
df["ratingRatio_log"] = np.log1p(df["RatingRatio"])
num_cols = ["pagesNumber", "PublishYear", "ratingRatio_log"]
scaler = RobustScaler()
X_num = scaler.fit_transform(df[num_cols].astype(float)).astype(np.float32)

# Title features
tfidf = TfidfVectorizer(stop_words="english", max_features=5000, ngram_range=(1,2), min_df=2)
X_title_sparse = tfidf.fit_transform(df["Name"].fillna(""))
svd = TruncatedSVD(n_components=256, random_state=42)
X_title = svd.fit_transform(X_title_sparse).astype(np.float32) * TITLE_WEIGHT

# Author features
hasher = FeatureHasher(n_features=512, input_type="string", alternate_sign=False)
author_tokens = df["AuthorId"].astype(str).apply(lambda a: [f"author={a}"]).tolist()
X_author = hasher.transform(author_tokens).toarray().astype(np.float32) * AUTHOR_WEIGHT

# Combine
X = np.hstack([X_num, X_author, X_title]).astype(np.float32)
faiss.normalize_L2(X)

# ==============================
# BUILD FAISS INDEX
# ==============================
d = X.shape[1]
index = faiss.IndexFlatIP(d)
index.add(X)

# ==============================
# VECTORIZE SINGLE BOOK
# ==============================
def vectorize_one_book(book_df: pd.DataFrame) -> np.ndarray:
    """Vectorize a single-row DataFrame into the feature space."""
    row = book_df.iloc[0]

    # Numeric
    pages = float(row["pagesNumber"])
    year = float(row["PublishYear"])
    rr = float(row["RatingRatio"])
    rr_log = np.log1p(rr)
    X_num_vec = scaler.transform(np.array([[pages, year, rr_log]], dtype=np.float32))

    # Author
    author_id = str(int(row["AuthorId"])) if "AuthorId" in row else "0"
    X_author_vec = hasher.transform([[f"author={author_id}"]]).toarray().astype(np.float32) * AUTHOR_WEIGHT

    # Title
    X_title_vec = svd.transform(tfidf.transform([row["Name"]])).astype(np.float32) * TITLE_WEIGHT

    # Combine
    vec = np.hstack([X_num_vec, X_author_vec, X_title_vec]).astype(np.float32)
    faiss.normalize_L2(vec)
    return vec

# ==============================
# RECOMMEND FROM LIST OF BOOKS
# ==============================
def recommend_from_books(book_dfs: list, k: int = 10):
    """
    book_dfs: list of single-row DataFrames
    Returns top-k recommended books not in the input list
    """
    # Vectorize all owned books
    vectors = [vectorize_one_book(b) for b in book_dfs]
    user_vector = np.mean(np.vstack(vectors), axis=0, keepdims=True)
    faiss.normalize_L2(user_vector)

    # Search
    D, I = index.search(user_vector, k + len(book_dfs))

    owned_titles = set([b.iloc[0]["Name"].lower() for b in book_dfs])
    recommended = 0

    print("\nUser owns:")
    for b in book_dfs:
        print(f"- {b.iloc[0]['Name']}")

    print("\nRecommended Books:")
    for row_idx, score in zip(I[0], D[0]):
        book_id = int(df.iloc[row_idx]["Id"])
        author_id = int(df.iloc[row_idx]["AuthorId"])
        title = bookid_to_name.get(book_id)
        author = authorid_to_name.get(author_id, "Unknown Author")

        if title.lower() in owned_titles:
            continue

        print(f"- {title} by {author} (Score: {score:.4f})")
        recommended += 1
        if recommended >= k:
            break

/tmp/ipython-input-12014/3151574079.py:23: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  kagglehub.load_dataset(


Using Colab cache for faster access to the 'goodreads-book-datasets-10m' dataset.


/tmp/ipython-input-12014/3151574079.py:23: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  kagglehub.load_dataset(


Using Colab cache for faster access to the 'goodreads-book-datasets-10m' dataset.

User owns:
- The Hobbit
- The Lord of the Rings
- Silmarillion

Recommended Books:
- Couples by K.E. Løgstrup (Score: 0.9990)
- Erections, Ejaculations, Exhibitions, and General Tales of Ordinary Madness by Michael O'Donoghue (Score: 0.9987)
- A Treasury of Kahlil Gibran by Don Abrams (Score: 0.9986)
- Buddenbrooks: The Decline of a Family by W. Douglas Robinson (Score: 0.9986)
- Holocaust by Unknown Author (Score: 0.9986)
- Montaillou: Cathars and Catholics in a French Village, 1294-1324 by Rick Mofina (Score: 0.9984)
- Ride a Pale Horse by Unknown Author (Score: 0.9983)
- Babe: The Legend Comes to Life by Unknown Author (Score: 0.9983)
- Caper by Mary Rourke (Score: 0.9982)
- The World as Will and Representation, Vol. 1 by Scott Thomas (Score: 0.9981)


In [ ]:
# ==============================
# EXAMPLE USAGE
# ==============================
book1 = pd.DataFrame({"Name":["The Hobbit"], "pagesNumber":[310], "PublishYear":[1937], "AuthorId":[0], "RatingRatio":[5000]})
book2 = pd.DataFrame({"Name":["The Lord of the Rings"], "pagesNumber":[1178], "PublishYear":[1954], "AuthorId":[0], "RatingRatio":[8000]})
book3 = pd.DataFrame({"Name":["Silmarillion"], "pagesNumber":[365], "PublishYear":[1977], "AuthorId":[0], "RatingRatio":[3000]})

recommend_from_books([book1, book2, book3], k=10)


User owns:
- The Hobbit
- The Lord of the Rings
- Silmarillion

Recommended Books:
- Couples by K.E. Løgstrup (Score: 0.9990)
- Erections, Ejaculations, Exhibitions, and General Tales of Ordinary Madness by Michael O'Donoghue (Score: 0.9987)
- A Treasury of Kahlil Gibran by Don Abrams (Score: 0.9986)
- Buddenbrooks: The Decline of a Family by W. Douglas Robinson (Score: 0.9986)
- Holocaust by Unknown Author (Score: 0.9986)
- Montaillou: Cathars and Catholics in a French Village, 1294-1324 by Rick Mofina (Score: 0.9984)
- Ride a Pale Horse by Unknown Author (Score: 0.9983)
- Babe: The Legend Comes to Life by Unknown Author (Score: 0.9983)
- Caper by Mary Rourke (Score: 0.9982)
- The World as Will and Representation, Vol. 1 by Scott Thomas (Score: 0.9981)


In [ ]:
# Hver bok er sin egen dataframe.